# Data Preparation

In [1]:
#__________________TASK 1 - Data prepration-___________________
import pandas as pd 
import sqlite3

# -----------------------------
# Step 1: Load excel data
# -----------------------------
df= pd.read_excel("fmcg_mult_prod_transactions_lifecycle_sample - Copy.xlsx")

#---------date time conversion---------#
df['transaction_month'] = pd.to_datetime(df['transaction_month'], format='%Y-%m')

# ------------------------Remove duplicate transactions--------------------
df.drop_duplicates(subset='transaction_id', inplace=True)

#-------------re-calculating sales amt for accuracy------
df['sales_amount']= df['unit_price']*df['volume_units']

#--------missing values-------
df.dropna(inplace=True)
# --------------------------------
# Step 3: Store in SQLite
# --------------------------------
conn = sqlite3.connect("sales_data.db")

df.to_sql(
    name="sales_data",
    con=conn,
    if_exists="replace",
    index=False
)

conn.commit()

#------checking the dataset again------#
df.info()
df.describe()
df.isnull().sum()

#Monthly Product-Level Aggregation
monthly_product_sales = (
    df.groupby(['transaction_month', 'product_id', 'product_name'])
      .agg(
          total_sales=('sales_amount', 'sum'),
          total_units=('volume_units', 'sum')
      )
      .reset_index()
)

#Sort for Time-Series Analysis
monthly_product_sales = monthly_product_sales.sort_values('transaction_month')


<class 'pandas.core.frame.DataFrame'>
Index: 27666 entries, 0 to 82946
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     27666 non-null  object        
 1   customer_number    27666 non-null  int64         
 2   transaction_month  27666 non-null  datetime64[ns]
 3   product_id         27666 non-null  object        
 4   product_name       27666 non-null  object        
 5   unit_price         27666 non-null  float64       
 6   volume_units       27666 non-null  int64         
 7   sales_amount       27666 non-null  float64       
 8   promo_offer        27666 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(2), object(4)
memory usage: 2.1+ MB


# Aggregation 

In [2]:
# Task Clean and transform data for analysis, including aggregation by time periods and product categories.

# ============================================
# TASK 1 (Part 2): Clean & Transform Data
# ============================================

import pandas as pd

# --------------------------------------------
# 1.Time-Based Feature Engineering
# --------------------------------------------

# Extract Year, Month, Quarter
df['year'] = df['transaction_month'].dt.year
df['month'] = df['transaction_month'].dt.month
df['quarter'] = df['transaction_month'].dt.quarter

# Create Year-Month Period (for monthly trend analysis)
df['year_month'] = df['transaction_month'].dt.to_period('M')


# --------------------------------------------
# 2.Monthly Aggregation
# --------------------------------------------

monthly_sales = (
    df.groupby('year_month')
      .agg(
          total_sales=('sales_amount', 'sum'),
          total_units=('volume_units', 'sum'),
          total_transactions=('transaction_id', 'nunique')
      )
      .reset_index()
)

# Convert Period to string for visualization / chatbot output
monthly_sales['year_month'] = monthly_sales['year_month'].astype(str)


# --------------------------------------------
# 3. Quarterly Aggregation
# --------------------------------------------

quarterly_sales = (
    df.groupby(['year', 'quarter'])
      .agg(
          total_sales=('sales_amount', 'sum'),
          total_units=('volume_units', 'sum'),
          total_transactions=('transaction_id', 'nunique')
      )
      .reset_index()
)


# --------------------------------------------
# 4.Product-Level Aggregation
# --------------------------------------------

product_sales = (
    df.groupby(['product_id', 'product_name'])
      .agg(
          total_sales=('sales_amount', 'sum'),
          total_units=('volume_units', 'sum'),
          total_transactions=('transaction_id', 'nunique')
      )
      .reset_index()
      .sort_values('total_sales', ascending=False)
)


# --------------------------------------------
# 5.Dynamic Product Category Creation
# --------------------------------------------

# If product_category does NOT exist, derive from product_name
df['product_category'] = df['product_name'].apply(lambda x: x.split()[0])


# --------------------------------------------
# 6.Category-Level Aggregation
# --------------------------------------------

category_sales = (
    df.groupby('product_category')
      .agg(
          total_sales=('sales_amount', 'sum'),
          total_units=('volume_units', 'sum'),
          total_transactions=('transaction_id', 'nunique')
      )
      .reset_index()
      .sort_values('total_sales', ascending=False)
)


# --------------------------------------------
# 7.Sort Monthly Data for Time-Series Analysis
# --------------------------------------------

monthly_sales = monthly_sales.sort_values('year_month')


# --------------------------------------------
# 8.Display Summary (Optional Validation)
# --------------------------------------------

print("Monthly Sales Preview:")
print(monthly_sales.head())

print("\nQuarterly Sales Preview:")
print(quarterly_sales.head())

print("\nTop Products Preview:")
print(product_sales.head())

print("\nCategory Sales Preview:")
print(category_sales.head())

Monthly Sales Preview:
  year_month  total_sales  total_units  total_transactions
0    2022-01    534857.89         1900                 334
1    2022-02   1045210.22         3826                 682
2    2022-03   1406513.60         5073                 934
3    2022-04   1697794.22         6368                1164
4    2022-05   1833233.72         6821                1260

Quarterly Sales Preview:
   year  quarter  total_sales  total_units  total_transactions
0  2022        1   2986581.71        10799                1950
1  2022        2   5445640.18        20286                3720
2  2022        3   6089698.49        22123                4036
3  2022        4   6005561.97        21768                3979
4  2023        1   6247501.69        22939                4154

Top Products Preview:
   product_id            product_name  total_sales  total_units  \
11       P012    Godrej Shampoo 200ml   2150511.59         3847   
34       P035  Milky Mist Paneer 200g   2100212.10         377

# Chatbot Core Analytics Engine

In [3]:
# ============================================
#Task 2:
#Section 1 Chatbot Core Analytics Engine
# ============================================

import sqlite3
import pandas as pd
from datetime import datetime
import time

# --------------------------------------------
# Database Path
# --------------------------------------------

DB_PATH = "sales_data.db"


# --------------------------------------------
# Function: Execute SQL Query
# --------------------------------------------

def execute_sql(query):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


# --------------------------------------------
# Function: Interpret User Query
# --------------------------------------------

def interpret_query(user_query):
    query = user_query.lower()

    # 1.Total Revenue
    if "total revenue" in query:
        return """
        SELECT ROUND(SUM(sales_amount), 2) AS total_revenue
        FROM sales_data;
        """

    # 2.Monthly Sales Trend
    elif "monthly" in query and "sales" in query:
        return """
        SELECT strftime('%Y-%m', transaction_month) AS month,
               ROUND(SUM(sales_amount), 2) AS total_sales
        FROM sales_data
        GROUP BY month
        ORDER BY month;
        """

    # 3.Top 5 Products
    elif "top" in query and "product" in query:
        return """
        SELECT product_name,
               ROUND(SUM(sales_amount), 2) AS total_sales
        FROM sales_data
        GROUP BY product_name
        ORDER BY total_sales DESC
        LIMIT 5;
        """

    # 4.Promotion Analysis
    elif "promotion" in query or "promo" in query:
        return """
        SELECT promo_offer,
               ROUND(SUM(sales_amount), 2) AS total_sales,
               SUM(volume_units) AS total_units
        FROM sales_data
        GROUP BY promo_offer;
        """

    else:
        return None


# ============================================
# Task 2 – Section 2
# Implement functions to determine question categories – whether it is a descriptive question or a visualization question
# ============================================

def classify_question(user_query):
    """
    Determines whether a user query is:
    - 'descriptive'
    - 'visualization'
    """

    query = user_query.lower().strip()

    visualization_keywords = [
        "show", "plot", "chart", "graph",
        "visualize", "trend", "compare",
        "display"
    ]

    if any(keyword in query for keyword in visualization_keywords):
        return "visualization"

    return "descriptive"

# ============================================
# Task 2 – Section 3
# Enable dynamic generation of textual insights based on the data obtained.
# ============================================

def generate_textual_insight(user_query, result_df):
    """
    Generates human-readable insights dynamically
    based on the query result DataFrame.
    """

    if result_df is None or result_df.empty:
        return "No data available to generate insights."

    query = user_query.lower()

    #Total Revenue Insight
    if "total revenue" in query:
        revenue = result_df.iloc[0, 0]
        return (
            f"The total revenue generated is ₹{revenue:,.2f}. "
            f"This reflects the overall financial performance of the retail store."
        )

    #Monthly Sales Insight
    elif "monthly" in query and "sales" in query:
        peak_row = result_df.loc[result_df.iloc[:, 1].idxmax()]
        return (
            f"Sales peaked in {peak_row.iloc[0]} with revenue of "
            f"₹{peak_row.iloc[1]:,.2f}. "
            f"This indicates the strongest performing month."
        )

    #Top Product Insight
    elif "top" in query and "product" in query:
        top_product = result_df.iloc[0]
        return (
            f"The top-performing product is '{top_product.iloc[0]}' "
            f"with total revenue of ₹{top_product.iloc[1]:,.2f}. "
            f"This product significantly contributes to overall sales."
        )

    #Promotion Insight
    elif "promotion" in query or "promo" in query:
        best_promo = result_df.loc[result_df.iloc[:, 1].idxmax()]
        return (
            f"The '{best_promo.iloc[0]}' promotion generated the highest revenue "
            f"of ₹{best_promo.iloc[1]:,.2f}. "
            f"This suggests promotional strategies positively impact sales."
        )

    return "Analysis completed successfully."



# Chatbot Core Analytics Engine (Section 2)

In [4]:
%%writefile query_engine.py

#==============================================================
# Section 2.1 – Backend Query Interpretation Logic
# Section 2.2 – Question Category Classification
# Section 2.3 – Dynamic Textual Insight Generation
# ==============================================================
import sqlite3

DB_PATH = "sales_data.db"

def execute_query(intent_data):
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # ----------------------------
    # Extract Intent Data
    # ----------------------------
    metric = intent_data.get("metric")
    years = intent_data.get("time_filter") or []
    quarter = intent_data.get("quarter")
    intent = intent_data.get("intent") or ""
    top_n = intent_data.get("top_n")

    # ----------------------------
    # Map Metric to Column
    # ----------------------------
    if metric == "sales":
        column = "sales_amount"
    elif metric == "customers":
        column = "customer_number"
    else:
        column = "sales_amount"

    query = ""
    params = []

    # ----------------------------
    # YEAR FILTER
    # ----------------------------
    year_condition = ""
    if years:
        if isinstance(years, list):
            placeholders = ",".join(["?"] * len(years))
            year_condition = f" AND strftime('%Y', transaction_month) IN ({placeholders})"
            params.extend(years)
        else:
            year_condition = " AND strftime('%Y', transaction_month) = ?"
            params.append(years)

    # ----------------------------
    # QUARTER FILTER
    # ----------------------------
    quarter_condition = ""
    if quarter:
        quarter_map = {
            "Q1": ("01", "02", "03"),
            "Q2": ("04", "05", "06"),
            "Q3": ("07", "08", "09"),
            "Q4": ("10", "11", "12"),
        }

        months = quarter_map.get(quarter)
        if months:
            quarter_condition = " AND strftime('%m', transaction_month) IN (?, ?, ?)"
            params.extend(months)

    # ==========================================================
    # SINGLE DECISION TREE (VERY IMPORTANT)
    # ==========================================================

    # 1️⃣ Comparison
    if intent == "comparison" and isinstance(years, list):
        query = f"""
        SELECT strftime('%Y', transaction_month) as year,
               SUM({column}) as total
        FROM sales_data
        WHERE 1=1 {year_condition}
        GROUP BY year
        ORDER BY year
        """

    # 2️⃣ Monthly Trend
    elif "trend" in intent:
        query = f"""
        SELECT strftime('%m', transaction_month) as month,
               SUM({column}) as total
        FROM sales_data
        WHERE 1=1 {year_condition}
        GROUP BY month
        ORDER BY month
        """

    # 3️⃣ Top Products
    elif intent == "ranking" and top_n:
        query = f"""
        SELECT product_name,
               SUM(sales_amount) as total
        FROM sales_data
        WHERE 1=1 {year_condition}
        GROUP BY product_name
        ORDER BY total DESC
        LIMIT ?
        """
        params.append(top_n)

    # 4️⃣ Customers
    elif metric == "customers":
        query = """
        SELECT COUNT(DISTINCT customer_number)
        FROM sales_data
        """

    # 5️⃣ Sales Total
    elif metric == "sales":
        query = f"""
        SELECT SUM({column})
        FROM sales_data
        WHERE 1=1 {year_condition} {quarter_condition}
        """

    # 6️⃣ Default
    else:
        query = f"""
        SELECT SUM({column})
        FROM sales_data
        WHERE 1=1 {year_condition} {quarter_condition}
        """

    # ----------------------------
    # Execute Query
    # ----------------------------
    cursor.execute(query, params)
    result = cursor.fetchall()

    conn.close()
    return result

#You’ve now built:

#Multi-year dynamic Sales Analytics Engine
#With NLP detection
#With quarter, trend, ranking support
#With comparison support

Writing query_engine.py


# GenAI Integration for Natural Language Interaction

In [5]:
%%writefile genai_engine.py

# ==============================================================
# Section 3.1 – GPT-4 API Integration
# Section 3.2 – Intent Extraction from User Query
# Section 3.3 – Contextual Response Generation
# ==============================================================


import json
import re
from datetime import datetime
import streamlit as st
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))




def classify_user_query(user_query):

    # ==========================================================
    # GPT-BASED INTENT EXTRACTION
    # ==========================================================
    try:
        system_prompt = """
        You are a Sales Analytics AI assistant.

        Extract structured JSON from user queries.

        Return ONLY valid JSON in this format:

        {
            "intent": "descriptive | comparison | trend | ranking",
            "metric": "sales | revenue | customers",
            "time_filter": "year if mentioned",
            "quarter": "Q1 | Q2 | Q3 | Q4 if mentioned",
            "top_n": "number if ranking"
        }

        Only return JSON. No explanation.
        """

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query}
            ],
            temperature=0
        )

        output_text = response.choices[0].message.content.strip()

        return json.loads(output_text)

    except Exception as e:
        print("⚠ GPT Failed, switching to rule-based fallback:", e)

    # ==========================================================
    # 2️⃣ RULE-BASED FALLBACK (Your Original Logic)
    # ==========================================================

    user_query = user_query.lower()
    current_year = datetime.now().year

    # Year Detection
    years = re.findall(r"\b20\d{2}\b", user_query)

    if "last year" in user_query:
        years.append(str(current_year - 1))
    if "this year" in user_query:
        years.append(str(current_year))

    # Month Detection
    months = [
        "january","february","march","april","may","june",
        "july","august","september","october","november","december"
    ]

    month_found = None
    for month in months:
        if month in user_query:
            month_found = month
            break

    # Quarter Detection
    quarter_match = re.search(r"\bq[1-4]\b", user_query)
    quarter = quarter_match.group().upper() if quarter_match else None

    # Top N Detection
    top_match = re.search(r"top\s*(\d+)", user_query)
    top_n = int(top_match.group(1)) if top_match else None

    # Highest Product
    if "highest" in user_query and "product" in user_query:
        return {
            "intent": "ranking",
            "metric": "sales",
            "top_n": 1,
            "time_filter": years[0] if years else None
        }

    # Comparison
    if "compare" in user_query:
        return {
            "intent": "comparison",
            "metric": "sales",
            "time_filter": years if years else None,
            "month": month_found,
            "quarter": quarter
        }

    # Trend
    elif "trend" in user_query:
        return {
            "intent": "trend",
            "metric": "sales",
            "time_filter": years[0] if years else None,
            "month": month_found,
            "quarter": quarter
        }

    # Ranking
    elif top_n:
        return {
            "intent": "ranking",
            "metric": "sales",
            "top_n": top_n,
            "time_filter": years[0] if years else None
        }

    # Customers
    elif "customer" in user_query:
        return {
            "intent": "descriptive",
            "metric": "customers",
            "time_filter": years[0] if years else None
        }

    # Revenue
    elif "revenue" in user_query:
        return {
            "intent": "descriptive",
            "metric": "revenue",
            "time_filter": years[0] if years else None,
            "month": month_found,
            "quarter": quarter
        }

    # Default Sales
    else:
        return {
            "intent": "descriptive",
            "metric": "sales",
            "time_filter": years[0] if years else None,
            "month": month_found,
            "quarter": quarter
        }




Writing genai_engine.py


# User Interface Development

In [6]:
%%writefile GENAI_Chatbot.py

# ==============================================================
# Streamlit Chat-Based Sales Analytics UI
# ==============================================================

import streamlit as st
import time
from datetime import datetime
import pandas as pd

from genai_engine import classify_user_query
from query_engine import execute_query

# --------------------------------------------------------------
# Page Config
# --------------------------------------------------------------
st.set_page_config(
    page_title="GenAI Sales Analytics Chatbot",
    page_icon="💬",
    layout="wide"
)

st.title("GenAI-Powered Sales Analytics Chatbot")

# --------------------------------------------------------------
# Session State Initialization
# --------------------------------------------------------------
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# --------------------------------------------------------------
# Suggested Questions Panel
# --------------------------------------------------------------
SUGGESTED_QUESTIONS = [
    "Show total sales for 2022",
    "Compare sales between 2022 and 2023",
    "Top 5 products by revenue",
    "Show monthly sales trend for 2023",
    "Total unique customers"
]

with st.sidebar:
    st.header("Suggested Queries")
    for q in SUGGESTED_QUESTIONS:
        if st.button(q):
            st.session_state.selected_query = q

# --------------------------------------------------------------
# Chat Input
# --------------------------------------------------------------
user_input = st.chat_input("Ask a business question")

query_to_process = None

if user_input:
    query_to_process = user_input.strip()
elif "selected_query" in st.session_state:
    query_to_process = st.session_state.selected_query
    del st.session_state.selected_query

# --------------------------------------------------------------
# Process Query
# --------------------------------------------------------------
if query_to_process:

    start_time = time.time()
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    try:
        intent_data = classify_user_query(query_to_process)
        result = execute_query(intent_data)
    except Exception as e:
        result = f"Database Error: {e}"
        intent_data = {}

    processing_time = round(time.time() - start_time, 2)

    # ✅ FIX 1: store intent
    st.session_state.chat_history.append({
        "timestamp": timestamp,
        "query": query_to_process,
        "intent": intent_data,
        "result": result,
        "processing_time_sec": processing_time
    })

# --------------------------------------------------------------
# Display Chat History (FIXED TABLE LOGIC)
# --------------------------------------------------------------
st.subheader("Chat History")

for chat in st.session_state.chat_history:
    with st.container():

        st.markdown(f"**Query:** {chat['query']}")
        st.markdown("**Result:**")

        result = chat["result"]
        intent = chat.get("intent", {})

        # -------- TABLE HANDLING FIX --------
        if isinstance(result, list) and result:

            # Case 1: list of dicts (already good)
            if isinstance(result[0], dict):
                st.table(pd.DataFrame(result))

            # Case 2: list of tuples/lists → intent-based columns
            elif isinstance(result[0], (list, tuple)):
                df = pd.DataFrame(result)

                metric = intent.get("metric", "").lower()
                intent_type = intent.get("intent", "").lower()

                if intent_type == "ranking":
                    if metric == "sales":
                        df.columns = ["Product", "Sales"]
                    elif metric == "revenue":
                        df.columns = ["Product", "Revenue"]
                    else:
                        df.columns = ["Item", "Value"]

                elif intent_type == "trend":
                    df.columns = ["Time Period", metric.title() or "Value"]

                elif intent_type == "comparison":
                    df.columns = ["Category", metric.title() or "Value"]

                elif metric == "customers":
                    df.columns = ["Category", "Customer Count"]

                else:
                    df.columns = [f"Column {i+1}" for i in range(df.shape[1])]

                st.table(df)

            else:
                st.write(result)

        elif isinstance(result, dict):
            st.table(pd.DataFrame([result]))

        else:
            st.write(result)

        st.caption(
            f"Executed on {chat['timestamp']} | "
            f"Processing time: {chat['processing_time_sec']} seconds"
        )

        st.divider()

# --------------------------------------------------------------
# Download Conversation Report
# --------------------------------------------------------------
if st.session_state.chat_history:
    st.subheader("Download Report")

    report_df = pd.DataFrame(st.session_state.chat_history)
    csv = report_df.to_csv(index=False).encode("utf-8")

    st.download_button(
        label="Download Chat & Analytics Report",
        data=csv,
        file_name="sales_analytics_chat_report.csv",
        mime="text/csv"
    )

Writing GENAI_Chatbot.py


# Automate the whole Pipeline

In [7]:
import subprocess
import sys
import time
def my_app():
    App_Path = "GENAI_Chatbot.py"
    return App_Path
App_Path = my_app()
process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", App_Path])